### Figure : Clustering UHVDB

In [ ]:
%%bash
# mkdir -p vclust
# cd vclust

# cp /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_creation/uhvdb.vclust_derep_reps.fna.gz \
#     .

# mv uhvdb.vclust_derep_reps.fna.gz \
#     uhvdb.fna.gz

# # cluster UHVDB at 95% ANI and 85% AF
# vclust \
#     prefilter \
#     --in uhvdb.fna.gz \
#     --out uhvdb.vclust_votu_prefilter.txt \
#     --threads 48 \
#     --min-ident 0.95

# vclust \
#     align \
#     --in uhvdb.fna.gz \
#     --out uhvdb.vclust_votu_ani.tsv \
#     --filter uhvdb.vclust_votu_prefilter.txt \
#     --threads 48 \
#     --out-ani 0.95 \
#     --out-qcov 0.85

# csvtk cut \
#     uhvdb.vclust_votu_ani.tsv \
#     --tabs \
#     --delete-header \
#     --fields query,reference,gani \
#     --out-file uhvdb.vclust_mcl.tsv

In [6]:
import polars as pl

(
    pl.read_csv('uhvdb_final_ids.txt', has_header=False, new_columns=['seq_header'])
        .with_columns([
            pl.col('seq_header').str.replace('^>', '').str.split(' ').list[0].alias('seq_name')
        ])[['seq_name']]
        .write_csv('uhvdb_final_ids.tsv', separator='\t', include_header=False)
)

In [ ]:
import polars as pl
uhvdb_final_ids = set(
    pl.read_csv('uhvdb_final_ids.tsv', has_header=False, new_columns=['seq_name'])['seq_name']
)

(
    pl.read_csv('vclust/uhvdb.vclust_mcl.tsv', separator='\t', has_header=False)
    .filter(
        (pl.col('column_1').is_in(uhvdb_final_ids)) & (pl.col('column_2').is_in(uhvdb_final_ids))
    )
    .write_csv('vclust/uhvdb.vclust_final_mcl.tsv', separator='\t', include_header=False)
)

# !mcl \
#     vclust/uhvdb.vclust_final_mcl.tsv \
#     --abc \
#     -sort revsize \
#     -te 32 \
#     -o vclust/uhvdb.vclust.mcl

In [ ]:
# 1. assign singletons and dtrs as reps
# 2. Run updated checkv completeness on remaining contings
# 3. assign remaining contigs to reps based on viral gene count and length vs expected length

In [42]:
# new uhgv votu selection script
import polars as pl

def load_mcl_clusters(mcl, unique):
    # assign sequences to mcl clusters
    clusters = {}

    cluster_id = 0
    with open(mcl, 'r') as mcl_file:
        for line in mcl_file:
            cluster_id += 1
            for node in line.strip().split():
                clusters[node] = cluster_id

    # assign unclustered sequences to their own cluster
    with open(unique, 'r') as unique_file:
        for line in unique_file:
            sequence = line.strip().split()[0]
            if sequence not in clusters:
                cluster_id += 1
                clusters[sequence] = cluster_id

    print("Number of clusters:", cluster_id)

    return clusters

def load_metadata(mine_report, uhgv_metadata, clusters):
    mine_report = (
        # load mine report and join with uhvdb metadata
        pl.read_csv(mine_report, separator='\t', columns=['seq_name', 'contig_length', 'proviral_length', 'viral_genes', 'completeness_method_2'], ignore_errors=True)
            .join(
                pl.read_csv(uhgv_metadata, separator='\t', columns=['uhgv_genome', 'genome_length', 'checkv_viral_markers', 'checkv_completeness_method'], ignore_errors=True),
                how='full', left_on='seq_name', right_on='uhgv_genome', suffix='_uhgv',
            )
            # retain only sequences that are in clusters
            .filter(
                (pl.col('seq_name').is_in(clusters.keys())) |
                (pl.col('uhgv_genome').is_in(clusters.keys()))
            )
            # create cluster_id and length columns
            .with_columns([
                pl.when(pl.col('seq_name').is_not_null())
                    .then(pl.col('seq_name'))
                    .otherwise(pl.col('uhgv_genome')).alias('contig_id'),
                pl.when(pl.col('viral_genes').is_not_null())
                    .then(pl.col('viral_genes'))
                    .otherwise(pl.col('checkv_viral_markers')).alias('viral_gene_count'),
                pl.when(pl.col('checkv_completeness_method').is_not_null())
                    .then(pl.col('checkv_completeness_method'))
                    .otherwise(pl.col('completeness_method_2')).alias('completeness_method'),
                pl.when(pl.col('contig_length').is_not_null())
                    .then(pl.col('contig_length'))
                    .when(pl.col('proviral_length').is_not_null())
                    .then(pl.col('proviral_length'))
                    .otherwise(pl.col('genome_length')).alias('length').cast(pl.Float64)
            ])
            .with_columns([pl.col('contig_id').replace_strict(clusters, default=None).alias('cluster_id')])
    )

    return mine_report

In [43]:
# vClust Cluster Reps
# 1. identify median length for each cluster
# 2. Assign singletons as vOTU reps
# 3. Assign longest DTRs (> median length) as vOTU reps
# 4. Assign linear genome with highest number of viral genes (tiebreaker: closest to expected AAI length) as vOTU reps
# 5. Output vOTU reps
# 6. Output vClust vOTU cluster information

# load cluster assignments
clusters = load_mcl_clusters('vclust/uhvdb.vclust.mcl', 'uhvdb_final_ids.tsv')

# load sequence metadata
mine_report = load_metadata(
    '/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/viruses.csvtk_concat.tsv',
    '/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_1/uhgv_metadata.tsv',
    clusters)


Number of clusters: 201946


In [44]:
# 1. calculate median length amd size each cluster
cluster_metrics = (
    mine_report.group_by('cluster_id').agg(
        [
            pl.col('length').median().alias('median_length'),
            pl.col('viral_gene_count').max().alias('max_viral_genes'),
            pl.col('contig_id').len().alias('num_seqs')
        ]
    )
)

In [45]:
cluster_info = (
    mine_report
        .join(cluster_metrics, on='cluster_id', how='inner')
        .filter(pl.col('cluster_id').is_not_null())
)

# 2. assign singletons as vOTU representatives
singleton_clusters = set(
    cluster_metrics.filter(pl.col('num_seqs') == 1)['cluster_id']
)
print("Number of singleton clusters:", len(singleton_clusters))

cluster_reps = (
    mine_report
        .filter(pl.col('cluster_id').is_in(singleton_clusters))['contig_id', 'cluster_id']
)

# 3. assign longest DTRs as vOTU representatives (if > median length)
dtr_cluster_reps = (
    cluster_info
        .filter(
            (
                (pl.col('completeness_method').str.contains('DTR'))
            ) &
            (~pl.col('cluster_id').is_in(cluster_reps['cluster_id']))
        )
        .filter(pl.col('genome_length') >= pl.col('median_length'))
        .sort('genome_length', descending=True)
        .group_by('cluster_id', maintain_order=True)
        .first()['contig_id', 'cluster_id']
)

print("Number of DTR cluster reps added:", dtr_cluster_reps.height)

cluster_reps = pl.concat([cluster_reps, dtr_cluster_reps])

# 4. Assign linear genome closest to expected AAI length with highest number of viral genes
linear_max_viral = (
    cluster_info
        .filter(
            (~pl.col('cluster_id').is_in(cluster_reps['cluster_id'])) &
            (pl.col('viral_gene_count') == pl.col('max_viral_genes'))
        )
)

# linear_max_viral[['contig_id']].write_csv('vclust/uhvdb.vclust_votu_rep_candidates.tsv', include_header=False)

# !seqkit grep \
#     uhvdb.rmdup.fna.gz \
#     --pattern-file vclust/uhvdb.vclust_votu_rep_candidates.tsv \
#     --out-file vclust/uhvdb.vclust_votu_rep_candidates.fna

# !seqkit split2 \
#     vclust/uhvdb.vclust_votu_rep_candidates.fna \
#     --by-size 10000 \
#     --out-dir checkv_split

# !sbatch checkv.sh

Number of singleton clusters: 157036
Number of DTR cluster reps added: 5216


In [46]:
# load checkv results
import glob

checkv_df_lst = []

for file in glob.glob('checkv.part*/completeness.tsv'):
    df = pl.read_csv(file, separator='\t', columns=['contig_id', 'aai_expected_length'], ignore_errors=True)
    checkv_df_lst.append(df)

checkv_df = pl.concat(checkv_df_lst)

# 4. Assign linear genome closest to expected AAI length with highest number of viral genes
linear_cluster_reps = (
    cluster_info
        .filter(
            (~pl.col('cluster_id').is_in(cluster_reps['cluster_id'])) &
            (pl.col('viral_gene_count') == pl.col('max_viral_genes'))
        )
        .join(checkv_df, how='left', on='contig_id')
        .with_columns([
            pl.col('aai_expected_length').cast(pl.String).str.replace('NA', pl.col('median_length')).cast(pl.Float64).alias('aai_expected_length'),
        ])
        .with_columns([
            (abs(pl.col('length').cast(pl.Float64) - pl.col('aai_expected_length').cast(pl.Float64))).alias('length_diff'),
        ])
        .sort(pl.col('length_diff'), descending=False)
        .group_by('cluster_id', maintain_order=True)
        .first()['contig_id', 'cluster_id']
)

print("Number of linear cluster reps added:", linear_cluster_reps.height)

cluster_reps = pl.concat([cluster_reps, linear_cluster_reps])

# 5. Output vOTU representatives
cluster_reps[['contig_id']].write_csv('vclust/uhvdb_vclust_votu_reps_final.tsv', include_header=False)

# 6. Output cluster information
(
    cluster_info
        .join(checkv_df, how='left', on='contig_id')
        [['contig_id', 'cluster_id', 'num_seqs', 'length', 'median_length', 'aai_expected_length', 'viral_gene_count', 'max_viral_genes', 'completeness_method']]
        .join(cluster_reps, on='cluster_id', how='full', suffix='_rep')
        .drop('cluster_id_rep')
        .rename({'contig_id_rep': 'votu_rep'})
        .write_csv('vclust/uhvdb_vclust_cluster_info_final.tsv', separator='\t')
)

Number of linear cluster reps added: 39694


In [ ]:
!seqkit grep \
    ../figure_s5/uhvdb_hq_derep_hc_final.fna.gz  \
    --pattern-file vclust/uhvdb_vclust_votu_reps_final.tsv \
    --out-file uhvdb.votu_reps.fna.gz